# 11 Product analysis from the shared dataset

**Questions:** Which product categories account for signed net sales, how much remains unclassified, and when were listed clearance SKUs sold?

Run 04 first. This notebook reads canonical lines and the shared product table. It joins categories only by exact SKU, preserving unknown products. It neither infers missing categories from SKU prefixes nor removes a SKU's entire history because it later appears on a clearance list.

The monetary population is all retained non-order-level lines across channels, including signed refunds. These are product-line exposures, not unique people or a causal comparison of first baskets.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # run from the notebooks folder

import pandas as pd
from IPython.display import display
from weavve.clean import load_dataset

dataset = load_dataset()
print("Shared dataset ID:", dataset.manifest["dataset_id"])
print("Local dataset:", dataset.path)
print("Transaction cutoff:", dataset.manifest["transaction_cutoff"])
print("Review flags:", len(dataset.manifest["warnings"]))
pd.set_option("display.max_columns", 20)


## Show exact-match coverage before category results

Order-level rows have no usable line-item identifier and are excluded from this product view. Product lines with missing or unmatched SKUs remain visible as unknown. Net sales and discounts use the shared corrected fields.


In [ ]:
lines = dataset.frame("order_lines")
product_lines = lines.loc[~lines["is_order_level"]].copy()
coverage = product_lines.groupby("product_match", dropna=False).agg(
    retained_lines=("source_row", "size"), signed_net_sales=("net", "sum"),
)
coverage["signed_net_sales"] = coverage["signed_net_sales"].map(float).round(2)
display(coverage.rename(index={True: "exact SKU match", False: "unknown or missing SKU"}))


## Category sales with unknowns retained

The product-master join must be many-to-one. Category labels come from the current master and may not describe historical products perfectly. The displayed net share is a signed-sales share, not a margin or unit-volume share.


In [ ]:
products = dataset.frame("products")
category_lookup = products[["clean_sku", "Category"]]
classified = product_lines.merge(category_lookup, left_on="sku", right_on="clean_sku", how="left", validate="many_to_one")
classified["analysis_category"] = classified["Category"].replace("", pd.NA).fillna("Unknown or unclassified")
category_summary = classified.groupby("analysis_category", dropna=False).agg(
    retained_lines=("source_row", "size"), signed_net_sales=("net", "sum"),
)
category_summary["signed_net_sales"] = category_summary["signed_net_sales"].map(float)
total = category_summary["signed_net_sales"].sum()
category_summary["signed_net_share_pct"] = 100 * category_summary["signed_net_sales"] / total if total else None
display(category_summary.sort_values("signed_net_sales", ascending=False).round(2))


## Clearance timing and discontinuation

The shared timing flag compares the transaction line date with the estimated quarter in File I. "Within" remains uncertain because the exact start date is absent. Discontinued SKUs are shown separately from clearance starts. A listed SKU sold before that quarter is not counted as a proven clearance sale.


In [ ]:
listed = product_lines.loc[product_lines["clearance_sku"]]
clearance_summary = listed.groupby(["clearance_status", "clearance_timing"], dropna=False).agg(
    retained_lines=("source_row", "size"), signed_net_sales=("net", "sum"),
)
clearance_summary["signed_net_sales"] = clearance_summary["signed_net_sales"].map(float).round(2)
display(clearance_summary)


## Before extending to first-basket recommendations

Specify the customer scope, first-order rule, product grouping and complete follow-up window before comparing long-run sales. Show unmatched-product coverage and first-order basket value alongside any later-sales association. This starter does not reproduce the report's alternative Table 11 classifications or prove that adding products causes future spending.
